# 2. Pull EIA Form 861

Downloads the annual EIA-861 zip (utility-level sales and customers) and unzips it into `data/raw/eia/YYYY/`. Skipped if that year is already there, since a final year's data doesn't change.

The EIA API only has state-level sales, so utility-level customer counts have to come from these files.

**Parameter:** `eia_year` (defaults to `EIA_YEAR` in `common.py`).

In [ ]:
eia_year = None

In [ ]:
import io
import zipfile

import requests

from common import EIA_YEAR, RAW_EIA_DIR, get_logger

log = get_logger("pull_eia")
eia_year = eia_year or EIA_YEAR

# Recent years are under zip/; older years move to archive/zip/.
EIA_URLS = [
    "https://www.eia.gov/electricity/data/eia861/zip/f861{year}.zip",
    "https://www.eia.gov/electricity/data/eia861/archive/zip/f861{year}.zip",
]

In [ ]:
out_dir = RAW_EIA_DIR / str(eia_year)
if (out_dir / f"Sales_Ult_Cust_{eia_year}.xlsx").exists():
    print(f"EIA-861 {eia_year} already in {out_dir}")
else:
    for url in (u.format(year=eia_year) for u in EIA_URLS):
        resp = requests.get(url, timeout=120)
        if resp.ok and resp.content[:2] == b"PK":  # a zip file, not an error page
            break
    else:
        raise RuntimeError(f"EIA-861 {eia_year} zip not found at {EIA_URLS}")
    out_dir.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(io.BytesIO(resp.content)).extractall(out_dir)
    log.info("downloaded %s to %s", url, out_dir)

sorted(p.name for p in out_dir.iterdir())